# Responses
`Starlette` 包含一些响应类，用于处理在发送通道上发回相应的 ASGI 消息。



In [ ]:
from starlette.responses import Response


async def app(scope, receive, send):
    assert scope['type'] == 'http'
    response = Response('Hello, world!', media_type='text/plain')
    await response(scope, receive, send)

`Starlette`中的响应是通过`Response`基类来提供的，但针对不同的响应还是提供了快捷的组建类。实例化一个`Response`类实例一般需要通过参数提供以下五种内容。

签名：: `Response(content, status_code=200, headers=None, media_type=None, background: BackgroundTask | None = None)`

- content，响应体内容。
- status_code，HTTP响应码。
- headers，响应头。
- media_type，响应的媒体类型。
- background，响应输出后要执行的后台任务。


`Starlette` 会自动包含一个 `Content-Length` 头信息。它还会根据媒体类型（`media_type`）加入一个 `Content-Type` 标头，并为文本类型添加一个字符集，除非媒体类型中已经指定了字符集。

实例化响应后，您可以通过将其作为 ASGI 应用程序实例调用来发送它。

### Set Cookie

`Response`实例可以设置客户端的`Cookie`，这主要通过`set_cookie()`和`delete_cookie()`两个方法实现。

**set_cookie()**     
签名:`Response.set_cookie(key, value, max_age=None, expires=None, path="/", domain=None, secure=False, httponly=False, samesite="lax", partitioned=False)`      

`starlette`提供了一种`set_cookie`方法，可让您在响应对象上设置·。     
- `key` - 将成为 cookie 键的字符串。
- `value` - 将作为 Cookie 值的字符串。
- `max_age` - 一个整数，用于定义 Cookie 的生命周期（以秒为单位）。负整数或值 0 将立即丢弃 Cookie。自选
- `expires` - 定义 Cookie 过期前的秒数的整数，或者 datetime。自选
- `path` - 一个字符串，用于指定 Cookie 将应用到的路由子集。自选
- `domain` - 一个字符串，用于指定 Cookie 对其有效的域。自选
- `secure` - 一个布尔值，指示仅当使用 SSL 和 HTTPS 协议发出请求时，Cookie 才会发送到服务器。自选
- `httponly` - 一个布尔值，指示无法通过 JavaScript 通过 Document.cookie 属性、XMLHttpRequest 或请求 API 访问 Cookie。自选
- `samesite` - 一个字符串，用于指定 Cookie 的 samesite 策略。有效值为 'lax'、'strict' 和 'none'。默认为 'lax'。自选
- `partitioned` - 一个布尔值，它向用户代理指示这些跨站点 Cookie 应仅在首次设置 Cookie 的同一顶级上下文中可用。仅适用于 Python 3.14+，否则将引发错误。自选



**Delete Cookie**


相反，`Starlette` 还提供了一个 `delete_cookie` 方法，用于手动过期已设置的 cookie。 签名：`Response.delete_cookie(key, path='/', domain=None)`






### 常用的响应类主要有以下这些。
- `HTMLResponse`，HTML文本响应类。
    - 获取一些文本或字节并返回 HTML 响应。
       ```python
        from starlette.responses import HTMLResponse
        async def app(scope, receive, send):
            assert scope['type'] == 'http'
            response = HTMLResponse('<html><body><h1>Hello, world!</h1></body></html>')
            await response(scope, receive, send)
       ```
- `PlainTextResponse`，纯文本响应类。
    - 接收一些文本或字节，并返回纯文本响应。
    ```python
        from starlette.responses import PlainTextResponse
        async def app(scope, receive, send):
            assert scope['type'] == 'http'
            response = PlainTextResponse('Hello, world!')
            await response(scope, receive, send)
    ```

- `JSONResponse`，返回application/json的数据响应类。
    - 获取一些数据并返回`application/json` 编码的响应。
    ```python
        from starlette.responses import JSONResponse
        async def app(scope, receive, send):
            assert scope['type'] == 'http'
            response = JSONResponse({'hello': 'world'})
            await response(scope, receive, send)
    ``` 
- `Custom JSON serialization` 自定义 JSON 序列化。
    - 如果需要对 JSON 序列化进行细粒度控制，可以子类化 `JSONResponse` 并重载 `render` 方法。
    - 例如，如果您想使用第三方 JSON 库，如 orjson：
    ```python
    from typing import Any
    import orjson
    from starlette.responses import JSONResponse
    class OrjsonResponse(JSONResponse):
        def render(self, content: Any) -> bytes:
            return orjson.dumps(content)
    ``` 
- `RedirectResponse`，产生302转向的响应类。
    - 返回 HTTP 重定向。默认使用 307 状态代码。
    ```python
    from starlette.responses import PlainTextResponse, RedirectResponse

    async def app(scope, receive, send):
    assert scope['type'] == 'http'
    if scope['path'] != '/':
        response = RedirectResponse(url='/')
    else:
        response = PlainTextResponse('Hello, world!')
    await response(scope, receive, send)
    ``` 
- `StreamingResponse`，产生流式数据的响应类。
    - 接收异步生成器或普通生成器/迭代器，并流式传输响应体。
    ```python
    from starlette.responses import StreamingResponse
    import asyncio

    async def slow_numbers(minimum, maximum):
        yield '<html><body><ul>'
        for number in range(minimum, maximum + 1):
            yield '<li>%d</li>' % number
            await asyncio.sleep(0.5)
        yield '</ul></body></html>'

    async def app(scope, receive, send):
        assert scope['type'] == 'http'
        generator = slow_numbers(1, 10)
        response = StreamingResponse(generator, media_type='text/html')
        await response(scope, receive, send)
    ``` 
    - 请注意，类文件对象（如由 open() 创建的对象）是普通的迭代器。因此，您可以在流式响应中直接返回它们。
- `FileResponse`，产生文件下载的响应类。
    - 异步流式传输文件作为响应。
    - 采用与其他响应类型不同的参数集来实例化：
        - `path` - 要流式传输的文件的文件路径。
        - `headers` - 要包含的任何自定义标头，作为字典。
        - `media_type` - 提供媒体类型的字符串。如果未设置，则文件名或路径将用于推断媒体类型。
        - `filename` — 如果设置，则此字段将包含在响应中 `Content-Disposition`.
        - `content_disposition_type` — 将包含在响应 `Content-Disposition` 中。可以设置为 “attachment” （默认） 或 “inline”。
    文件响应将包括适当的 `Content-Length`、`Last-Modified` 和 `ETag` 标头。
    
    ```python
    from starlette.responses import FileResponse
    async def app(scope, receive, send):
        assert scope['type'] == 'http'
        response = FileResponse('statics/favicon.ico')
        await response(scope, receive, send)

    ``` 
    - 如果文件存在，响应中将包含 `Accept-Ranges: bytes` 标头。目前只支持字节范围单位。
    - 如果请求包含 "范围 "标头，且文件存在，则响应将是 206 "部分内容 "响应，其中包含请求的字节范围。如果范围无效，响应将是一个 416 范围无法满足的响应。

- `EventSourceResponse` [第三方回应](https://github.com/sysid/sse-starlette)
    - 实现服务器发送事件的响应类。它能实现从服务器到客户端的事件流，而无需复杂的 websockets。


